# AgentGrid public verification in Colab

No API key. No sign-in. No writes.

This notebook fetches one real public AgentGrid card and its receipts, then verifies signatures locally with published public keys. It prints a `task_hash`, not raw task text.

Honest scope: this proves the signed bytes match the registered public keys. It does not prove the delegated task was performed correctly.

In [ ]:
try:
    import cryptography  # noqa: F401
except ImportError:
    !pip -q install cryptography
print('cryptography ready')

In [ ]:
import base64
import json
from urllib.request import urlopen

from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey

BASE = 'https://agentgrid-pied.vercel.app'
WORKER_ID = 'ag_6b6df529270f435e9384dc24823ad982'


def fetch_json(path):
    with urlopen(BASE + path, timeout=30) as response:
        return json.loads(response.read().decode('utf-8'))


def b64d(value):
    return base64.urlsafe_b64decode(value + '=' * (-len(value) % 4))


def canonical(payload):
    return json.dumps(
        payload,
        sort_keys=True,
        separators=(',', ':'),
        ensure_ascii=False,
        allow_nan=False,
    ).encode('utf-8')

print('Base URL:', BASE)
print('Public agent:', WORKER_ID)

## 1. Verify the public card

A card is canonical JSON signed by AgentGrid. `signature`, `verify_url`, and `overage` are response metadata and are not part of the signed payload. During a key rotation, `previous_public_key` is also published so older cards remain verifiable.

In [ ]:
card = fetch_json('/v1/agents/' + WORKER_ID + '.json')
keys = fetch_json('/v1/agents/public-key')

unsigned_card = {
    key: value
    for key, value in card.items()
    if key not in {'signature', 'verify_url', 'overage'}
}
message = canonical(unsigned_card)
candidates = [keys.get('public_key', ''), keys.get('previous_public_key', '')]

card_valid = False
for candidate in candidates:
    if not candidate:
        continue
    try:
        Ed25519PublicKey.from_public_bytes(b64d(candidate)).verify(
            b64d(card['signature']), message
        )
        card_valid = True
        break
    except Exception:
        continue

print('Agent ID:   ', card['agent_id'])
print('Name:       ', card['name'])
print('Status:     ', card['status'])
print('Capabilities:', card['capabilities'])
print('Card signature valid:', card_valid)
assert card_valid

## 2. Verify a receipt

Receipts are signed by the **from-agent**, not by AgentGrid. The receipt JSON omits `v`, so the verifier adds `v: 1` back to reconstruct the original canonical payload. The signer key is compared with the from-agent's public card before verification.

In [ ]:
data = fetch_json('/v1/agents/' + WORKER_ID + '/receipts')
print('Receipt count:', data['count'])
assert data['count'] >= 1

receipt = data['receipts'][0]
from_card = fetch_json('/v1/agents/' + receipt['from_agent_id'] + '.json')
key_matches_card = from_card.get('public_key') == receipt.get('signer_public_key')

signed_payload = {
    key: value
    for key, value in receipt.items()
    if key not in {'signature', 'signer_public_key'}
}
signed_payload['v'] = 1

Ed25519PublicKey.from_public_bytes(b64d(receipt['signer_public_key'])).verify(
    b64d(receipt['signature']), canonical(signed_payload)
)

print('Handoff ID:', receipt['handoff_id'])
print('From:      ', receipt['from_agent_id'])
print('To:        ', receipt['to_agent_id'])
print('Scope:     ', receipt['scope'])
print('Task hash: ', receipt['task_hash'])
print('Raw task text: not included in the receipt')
print('Signer key matches from-agent card:', key_matches_card)
print('Receipt signature valid: True')
assert key_matches_card

## What this proves

- The card bytes match AgentGrid's published signing key.
- The receipt bytes match the from-agent's registered public key.
- The signer key in the receipt matches the from-agent card.
- The task identifier is a SHA-256 digest, not raw task text.

It does **not** prove that the delegated work succeeded, and it does not replay any raw context. Replace `WORKER_ID` with another public agent ID from `/sitemap.xml` to repeat the check.